# Generador de clientes sintéticos pensionarios de México

Crea una base de personas ficticias (IMSS Ley 73 / Ley 97, ISSSTE Cuentas Individuales e independientes sin afiliación) a partir de los CSV del directorio de entrada. No utiliza ni produce datos personales reales.

- **Parámetros:** todo lo ajustable (rutas, semilla, `N_CLIN`, fecha de corte, proporción por perfil, distribuciones de edad, salario, saldos, probabilidades y montos legales de referencia) vive en `config/parametros.py`.
- **Lógica:** paquete `generador/`. Cada cliente se genera por perfil, se clasifica con reglas pensionarias y se valida al 100 %.
- `N_CLIN` es exactamente el número de clientes a generar.
- Los importes están en pesos constantes de la fecha de corte. Todas las resoluciones son simuladas.

In [16]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona si Jupyter se abre desde la raíz del proyecto o desde esta carpeta.
RUTA_NOTEBOOK = next(
    (ruta for ruta in (Path.cwd(), Path.cwd() / 'SyntethicDataGenerator') if (ruta / 'config').is_dir()),
    None,
)
if RUTA_NOTEBOOK is None:
    raise FileNotFoundError('No se encontró el directorio config junto al notebook.')
sys.path.insert(0, str(RUTA_NOTEBOOK))

# Descarta módulos en caché para que los cambios en config/ se apliquen al volver a ejecutar.
for modulo in [m for m in sys.modules if m.split('.')[0] in ('config', 'generador')]:
    del sys.modules[modulo]

import config as cfg
from generador import generar_clientes

cfg.DIR_OUTPUT.mkdir(exist_ok=True)
rng = np.random.default_rng(cfg.SEMILLA)

In [17]:
nombres = pd.read_csv(cfg.ARCHIVO_NOMBRES, encoding=cfg.CODIFICACION_ENTRADA)
apellidos = pd.read_csv(cfg.ARCHIVO_APELLIDOS, encoding=cfg.CODIFICACION_ENTRADA)

# Controles de calidad de los insumos.
assert len(nombres) == cfg.NOMBRES_ESPERADOS, f'Se esperan {cfg.NOMBRES_ESPERADOS} nombres.'
assert set(nombres['genero']) == set(cfg.GENEROS_ARCHIVO), 'Géneros inesperados en el archivo de nombres.'
assert len(apellidos) == cfg.APELLIDOS_ESPERADOS, f'Se esperan {cfg.APELLIDOS_ESPERADOS} apellidos.'

display(nombres.groupby('genero').size().rename('cantidad'))
display(apellidos.head())

genero
Hombre    250
Mujer     250
Name: cantidad, dtype: int64

,id,apellido
0,1,Hernández
1,2,García
2,3,Martínez
3,4,López
4,5,González


In [18]:
# Genera historial laboral, saldos (salario, RCV, AV, vivienda), pensión estimada e identificadores.
clientes = generar_clientes(rng, nombres, apellidos)
clientes.head(10)

,id_cliente_sintetico,tipo_cliente,nombre,apellido_paterno,apellido_materno,nombre_completo,genero,fecha_nacimiento,edad,estado_nacimiento,...,saldo_pensionario_utilizable,beneficiarios,modalidad_pension,pension_autofinanciable_estimada,pension_minima_legal_aplicable,pension_final_estimada,resultado_pensionario,motivo_resultado,resolucion_simulada,validacion_reglas
0,1,04,Claudio,Treviño,Escamilla,Claudio Treviño Escamilla,H,1962-06-30,64,Aguascalientes,...,848971.77,NaN,NaN,15612.25,9577.22,15612.25,PENSION_ORDINARIA,Cuantía Ley 73 superior a la pensión mínima le...,True,OK
1,2,09,Emmanuel,Domínguez,Meza,Emmanuel Domínguez Meza,H,1958-06-18,68,Zacatecas,...,217296.49,SI,RETIRO_PROGRAMADO,1374.76,NaN,0.00,NEGATIVA_PENSION,ISSSTE con 17.41 años; se requieren 25.,True,OK
2,3,08,Esperanza,Ortega,Del Río,Esperanza Ortega Del Río,M,1966-09-20,60,Querétaro,...,807417.91,SI,RENTA_VITALICIA,3478.79,6000.00,6000.00,PENSION_GARANTIZADA,"Cumple requisitos; saldo insuficiente, aplica ...",True,OK
3,4,05,Emely,Labastida,Alvarado,Emely Labastida Alvarado,M,1964-02-12,62,Baja California,...,191762.83,NaN,NaN,7237.80,9577.22,9577.22,PENSION_MINIMA_LEY_73,Cuantía Ley 73 inferior al mínimo; se aplica e...,True,OK
4,5,01,Joel,Escobedo,Barajas,Joel Escobedo Barajas,H,1963-12-12,62,Quintana Roo,...,1072632.64,SI,RENTA_VITALICIA,5309.60,4662.24,5309.60,PENSION_RECURSOS_PROPIOS,Saldo suficiente para financiar la pensión por...,True,OK
5,6,05,Adán,Ugalde,Palacios,Adán Ugalde Palacios,H,1966-02-05,60,Tamaulipas,...,205305.02,NaN,NaN,6695.09,9577.22,9577.22,PENSION_MINIMA_LEY_73,Cuantía Ley 73 inferior al mínimo; se aplica e...,True,OK
6,7,02,Edgar,Ortiz,Barrientos,Edgar Ortiz Barrientos,H,1961-02-26,65,Querétaro,...,310380.12,SI,RETIRO_PROGRAMADO,1759.01,4412.32,4412.32,PENSION_GARANTIZADA,"Cumple requisitos; saldo insuficiente, aplica ...",True,OK
7,8,03,Imelda,Godoy,Escobedo,Imelda Godoy Escobedo,M,1966-05-11,60,Aguascalientes,...,147583.63,SI,RENTA_VITALICIA,635.87,NaN,0.00,NEGATIVA_PENSION,Ley 97 con 597 semanas; se requieren 875.,True,OK
8,9,07,Leonel,Valdez,Marín,Leonel Valdez Marín,H,1965-01-09,61,Durango,...,1782417.29,SI,RENTA_VITALICIA,8585.97,6000.00,8585.97,PENSION_RECURSOS_PROPIOS,Saldo suficiente para financiar la pensión por...,True,OK
9,10,02,Josué,Gutiérrez,Ponce,Josué Gutiérrez Ponce,H,1957-01-22,69,San Luis Potosí,...,467121.32,SI,RENTA_VITALICIA,2949.87,5086.40,5086.40,PENSION_GARANTIZADA,"Cumple requisitos; saldo insuficiente, aplica ...",True,OK


In [19]:
# Control de calidad: el 100 % de los registros debe cumplir las reglas.
invalidos = clientes[clientes['validacion_reglas'] != 'OK']
assert invalidos.empty, f'{len(invalidos)} registros incumplen reglas:\n{invalidos[["id_cliente_sintetico", "validacion_reglas"]].head()}'
assert clientes['id_cliente_sintetico'].is_unique and clientes['CURP'].is_unique
assert len(clientes) == cfg.N_CLIN

conteos = {
    'tipo de cliente': ['tipo_cliente', 'resultado_pensionario'],
    'institución y régimen': ['institucion', 'regimen_pensionario'],
}
for titulo, columnas in conteos.items():
    print(titulo)
    display(clientes.fillna({'institucion': cfg.NO_APLICA}).groupby(columnas).size().rename('clientes'))

cotizantes = clientes[clientes['institucion'].notna()]
display(pd.crosstab(pd.cut(clientes['edad'], [0, 49, 59, 64, 69, 120], labels=['<50', '50-59', '60-64', '65-69', '70+']),
                    clientes['tipo_cliente']))
display(cotizantes.groupby('tipo_cliente')[['semanas_cotizadas', 'anios_cotizados']].describe().round(1))
display(clientes[['salario_mensual', 'saldo_rcv', 'saldo_av', 'saldo_vivienda']].describe(percentiles=[.5, .9, .99]).round(0))
print(f"Clientes con AV = 0: {(cotizantes['saldo_av'] == 0).mean():.1%} | con vivienda = 0: {(cotizantes['saldo_vivienda'] == 0).mean():.1%}")
display(cotizantes[['salario_mensual', 'semanas_cotizadas', 'saldo_rcv', 'saldo_av', 'saldo_vivienda']].corr('spearman').round(2))

tipo de cliente


tipo_cliente  resultado_pensionario     
01            PENSION_RECURSOS_PROPIOS       8000
02            PENSION_GARANTIZADA           17000
03            NEGATIVA_PENSION               8000
04            PENSION_ORDINARIA             15000
05            PENSION_MINIMA_LEY_73         10000
06            NEGATIVA_PENSION               6000
07            PENSION_RECURSOS_PROPIOS      10000
08            PENSION_GARANTIZADA           10000
09            NEGATIVA_PENSION               6000
10            SIN_AFILIACION_PENSIONARIA    10000
Name: clientes, dtype: int64

institución y régimen


institucion  regimen_pensionario        
IMSS         IMSS_LEY_73                    31000
             IMSS_LEY_97                    33000
ISSSTE       ISSSTE_CUENTAS_INDIVIDUALES    26000
NO_APLICA    NO_APLICA                      10000
Name: clientes, dtype: int64

tipo_cliente,01,02,03,04,05,06,07,08,09,10
edad,,,,,,,,,,
<50,0,0,0,0,0,0,0,0,0,4290
50-59,0,0,0,0,0,0,0,0,0,3894
60-64,6314,12660,5965,9641,9922,4182,7784,8883,0,1194
65-69,1686,4285,1999,5218,78,1757,2178,1115,5262,622
70+,0,55,36,141,0,61,38,2,738,0


semanas_cotizadas                                                 \
                         count    mean    std     min     25%     50%     75%   
tipo_cliente                                                                    
01                      8000.0  1058.7  117.4   875.0   964.0  1045.0  1138.0   
02                     17000.0  1060.0  117.7   875.0   965.0  1046.0  1140.0   
03                      8000.0   493.3  142.0   102.0   395.0   494.0   594.0   
04                     15000.0  1605.8  201.9   917.0  1469.0  1601.5  1739.2   
05                     10000.0  1432.9  169.3   868.0  1315.0  1432.0  1547.0   
06                      6000.0   296.3   90.6    51.0   232.0   297.0   362.0   
07                     10000.0  1481.9  104.2  1304.0  1402.0  1473.0  1551.0   
08                     10000.0  1432.6   80.0  1304.0  1370.0  1424.0  1485.0   
09                      6000.0   923.0  193.2   246.0   789.0   935.0  1067.0   

                     anios_cotizados                                           
                 max           count  mean  std   min   25%   50%   75%   max  
tipo_cliente                                                                   
01            1488.0          8000.0  20.3  2.2  16.8  18.5  20.0  21.8  28.5  
02            1492.0         17000.0  20.3  2.3  16.8  18.5  20.0  21.9  28.6  
03             870.0          8000.0   9.5  2.7   2.0   7.6   9.5  11.4  16.7  
04            2366.0         15000.0  30.8  3.9  17.6  28.2  30.7  33.3  45.4  
05            2065.0         10000.0  27.5  3.2  16.6  25.2  27.4  29.6  39.6  
06             498.0          6000.0   5.7  1.7   1.0   4.4   5.7   7.0   9.6  
07            1872.0         10000.0  28.4  2.0  25.0  26.9  28.2  29.7  35.9  
08            1768.0         10000.0  27.5  1.5  25.0  26.3  27.3  28.5  33.9  
09            1299.0          6000.0  17.7  3.7   4.7  15.1  17.9  20.5  24.9

,salario_mensual,saldo_rcv,saldo_av,saldo_vivienda
count,100000.0,100000.0,100000.0,100000.0
mean,20892.0,413526.0,18343.0,132089.0
std,11858.0,377417.0,111311.0,168805.0
min,9582.0,0.0,0.0,0.0
50%,17759.0,324153.0,0.0,70917.0
90%,34922.0,919641.0,17808.0,370846.0
99%,63333.0,1733348.0,457595.0,704759.0
max,83555.0,4122612.0,5833928.0,1224236.0


Clientes con AV = 0: 86.8% | con vivienda = 0: 28.2%


,salario_mensual,semanas_cotizadas,saldo_rcv,saldo_av,saldo_vivienda
salario_mensual,1.00,-0.11,0.65,0.22,0.25
semanas_cotizadas,-0.11,1.00,0.37,-0.03,0.24
saldo_rcv,0.65,0.37,1.00,0.17,0.32
saldo_av,0.22,-0.03,0.17,1.00,0.07
saldo_vivienda,0.25,0.24,0.32,0.07,1.00


In [20]:
archivo_salida = cfg.DIR_OUTPUT / cfg.PLANTILLA_ARCHIVO_SALIDA.format(n=cfg.N_CLIN)
clientes.to_csv(archivo_salida, index=False, encoding=cfg.CODIFICACION_SALIDA, date_format=cfg.FORMATO_FECHA)

print(f'{len(clientes):,} clientes generados en: {archivo_salida.resolve()}')

100,000 clientes generados en: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/clientes_sinteticos_100000.csv


In [21]:
clientes.head()


,id_cliente_sintetico,tipo_cliente,nombre,apellido_paterno,apellido_materno,nombre_completo,genero,fecha_nacimiento,edad,estado_nacimiento,...,saldo_pensionario_utilizable,beneficiarios,modalidad_pension,pension_autofinanciable_estimada,pension_minima_legal_aplicable,pension_final_estimada,resultado_pensionario,motivo_resultado,resolucion_simulada,validacion_reglas
0,1,04,Claudio,Treviño,Escamilla,Claudio Treviño Escamilla,H,1962-06-30,64,Aguascalientes,...,848971.77,NaN,NaN,15612.25,9577.22,15612.25,PENSION_ORDINARIA,Cuantía Ley 73 superior a la pensión mínima le...,True,OK
1,2,09,Emmanuel,Domínguez,Meza,Emmanuel Domínguez Meza,H,1958-06-18,68,Zacatecas,...,217296.49,SI,RETIRO_PROGRAMADO,1374.76,NaN,0.00,NEGATIVA_PENSION,ISSSTE con 17.41 años; se requieren 25.,True,OK
2,3,08,Esperanza,Ortega,Del Río,Esperanza Ortega Del Río,M,1966-09-20,60,Querétaro,...,807417.91,SI,RENTA_VITALICIA,3478.79,6000.00,6000.00,PENSION_GARANTIZADA,"Cumple requisitos; saldo insuficiente, aplica ...",True,OK
3,4,05,Emely,Labastida,Alvarado,Emely Labastida Alvarado,M,1964-02-12,62,Baja California,...,191762.83,NaN,NaN,7237.80,9577.22,9577.22,PENSION_MINIMA_LEY_73,Cuantía Ley 73 inferior al mínimo; se aplica e...,True,OK
4,5,01,Joel,Escobedo,Barajas,Joel Escobedo Barajas,H,1963-12-12,62,Quintana Roo,...,1072632.64,SI,RENTA_VITALICIA,5309.60,4662.24,5309.60,PENSION_RECURSOS_PROPIOS,Saldo suficiente para financiar la pensión por...,True,OK
